In [7]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from huggingface_hub import get_token
from transformers import pipeline

MODEL_ID = "google/medgemma-4b-it"
INPUT_PATH = Path(<PATH>)
OUTPUT_PATH = Path(<OUTPUT>)

print(INPUT_PATH.exists())

True


In [2]:

def build_prompt(row):
    input_text = row.get("input_text")
    if pd.notna(input_text) and str(input_text).strip():
        case_text = str(input_text).strip()
    else:
        case_text = (
            f"Patient sex: {row.get('GENDER', 'unknown')}\n"
            f"Admission type: {row.get('ADMISSION_TYPE', 'unknown')}\n"
            f"Admission diagnosis: {row.get('DIAGNOSIS', 'unknown')}\n\n"
            f"Laboratory test: {row.get('lab_name', 'unknown')}\n"
            f"Fluid: {row.get('fluid', 'unknown')}\n"
            f"Category: {row.get('category', 'unknown')}\n"
            f"Measured value: {row.get('VALUE', 'unknown')} {row.get('VALUEUOM', '')}\n"
            f"Abnormal flag: {row.get('FLAG', 'not specified')}"
        )

    return f"""You are generating educational clinical explanations for laboratory results.

Given the laboratory result below, write a short explanation in 2-3 sentences.

Rules:
- Use cautious medical language.
- Do not make a final diagnosis.
- Do not recommend treatment.
- Do not invent missing information.
- Mention that the result should be interpreted in clinical context.

Input:
{case_text}

Output:
"""



In [3]:


def main():
    token = <TOKEN>

    if not token:
        sys.exit(
            "Missing Hugging Face token. Put HF_TOKEN=your_token in .env, "
            "run hf_login.py, or set HF_TOKEN in your terminal."
        )

    if not INPUT_PATH.exists():
        sys.exit(f"Missing input file: {INPUT_PATH}")

    df = pd.read_csv(INPUT_PATH).head(20).copy()
    print(f"Loaded {len(df)} rows from {INPUT_PATH}", flush=True)

    if not torch.cuda.is_available():
        print(
            "CUDA GPU is not available. MedGemma 4B may download slowly and run "
            "very slowly on CPU.",
            flush=True,
        )

    try:
        print(f"Loading {MODEL_ID}...", flush=True)
        generator = pipeline(
            "text-generation",
            model=MODEL_ID,
            token=token,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        print("Model loaded. Generating outputs...", flush=True)
    except OSError as exc:
        if "gated repo" in str(exc).lower() or "403" in str(exc):
            sys.exit(
                f"Cannot access {MODEL_ID}. Visit https://huggingface.co/{MODEL_ID} "
                "and request/accept access for the Hugging Face account used by "
                "your saved login or HF_TOKEN."
            )
        raise

    outputs = []
    for idx, row in df.iterrows():
        prompt = build_prompt(row)
        print(f"Generating {len(outputs) + 1}/20: {row.get('lab_name', 'unknown')}", flush=True)
        result = generator(prompt, max_new_tokens=80, do_sample=False)
        generated_text = result[0]["generated_text"]
        answer = generated_text[len(prompt):].strip()
        outputs.append(answer)
        df.loc[idx, "medgemma_output"] = answer
        df.to_csv(OUTPUT_PATH, index=False)
        print(f"Saved progress to {OUTPUT_PATH}", flush=True)

    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved: {OUTPUT_PATH.resolve()}")



In [4]:
from huggingface_hub import get_token, login, whoami
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
from google.colab import files

uploaded = files.upload()

Saving mimic_labs_20_for_testing.csv to mimic_labs_20_for_testing.csv


In [ ]:
main()

Loaded 20 rows from mimic_labs_20_for_testing.csv
Loading google/medgemma-4b-it...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved progress to medgemma_20_outputs.csv
Generating 14/20: Sodium


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved progress to medgemma_20_outputs.csv
Generating 15/20: Urea Nitrogen


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
